In [11]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import scienceplots
import torchvision
from datasets import load_dataset, Image
import torchvision.transforms.v2 as transforms_v2

torch.manual_seed(67)
np.random.seed(67)
sns.set_palette("husl")
# importing my preferred style
plt.style.use(['science', 'no-latex', 'grid'])
plt.rcParams.update({'font.size':20})
plt.rcParams['figure.figsize'] = (10,10)
plt.rcParams['axes.formatter.use_mathtext'] = True

In [12]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

print(f"Is CUDA supported by this system? {torch.cuda.is_available()}")
print(f"Current CUDA device name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"CUDA version: {torch.version.cuda}")
torch.backends.cudnn.benchmark = True

Using cuda device
Is CUDA supported by this system? True
Current CUDA device name: NVIDIA GeForce RTX 3090
CUDA version: 13.0


In [13]:
from torchvision import datasets, transforms

# CIFAR-10 dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset_full = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Split train into train and validation
from torch.utils.data import random_split
train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size
train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])


In [14]:
# check out dataset
print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"Classes: {train_dataset_full.classes}")
print(f"Sample: {train_dataset[0]}")

Train dataset size: 40000
Val dataset size: 10000
Test dataset size: 10000
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Sample: (tensor([[[-0.0902, -0.1686, -0.2157,  ..., -0.2706, -0.2784, -0.3255],
         [-0.0902, -0.1608, -0.2471,  ..., -0.3098, -0.3569, -0.3961],
         [-0.1922, -0.2314, -0.2941,  ..., -0.2549, -0.3176, -0.3961],
         ...,
         [ 0.0118, -0.0824,  0.0196,  ...,  0.2078,  0.2549,  0.1765],
         [ 0.2549,  0.2000,  0.2314,  ...,  0.2549,  0.2784,  0.2706],
         [ 0.3176,  0.2941,  0.2941,  ...,  0.1451,  0.1922,  0.2078]],

        [[ 0.0353, -0.0196, -0.0510,  ..., -0.1922, -0.1922, -0.2157],
         [ 0.0510, -0.0196, -0.0980,  ..., -0.2314, -0.2392, -0.2549],
         [-0.0196, -0.0510, -0.1608,  ..., -0.2078, -0.2000, -0.2392],
         ...,
         [ 0.1608,  0.0039,  0.1059,  ...,  0.4588,  0.5137,  0.4431],
         [ 0.4667,  0.3882,  0.4196,  ...,  0.4902,  0.5216,  0.5294],
     

In [15]:
images_train = torch.stack( [ sample[0] for sample in train_dataset ] , dim = 3)
images_train.shape
images_valid = torch.stack( [ sample[0] for sample in val_dataset ] , dim = 3)
images_valid.shape
images_test = torch.stack( [ sample[0] for sample in test_dataset ] , dim = 3)
images_test.shape

torch.Size([3, 32, 32, 10000])

In [16]:
train_mean = images_train.view(3, -1).mean(dim=1)
train_std = images_train.view(3, -1).std(dim=1)
valid_mean = images_valid.view(3, -1).mean(dim=1)
valid_std = images_valid.view(3, -1).std(dim=1)
test_mean = images_test.view(3, -1).mean(dim=1)
test_std = images_test.view(3, -1).std(dim=1)

In [17]:
from torch.utils.data import Dataset
import random
from torchvision.transforms import v2

class AugmentedDataset(Dataset):
    def __init__(self, data, toaugment=False):
        self.data = data
        self.toaugment = toaugment
        self.transforms = [
            lambda x: x,  # original
            v2.functional.hflip,  # horizontal flip
            v2.functional.vflip,  # vertical flip
            v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # color jitter
            v2.RandomRotation(degrees=15),  # random rotation
            v2.RandomCrop(size=(32, 32), padding=4),  # random crop with padding
            v2.Compose([  # horizontal flip + color jitter + rotation
                v2.functional.hflip,
                v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
                v2.RandomRotation(degrees=15)
            ]),
            v2.Compose([  # vertical flip + crop
                v2.functional.vflip,
                v2.RandomCrop(size=(32, 32), padding=4)
            ])
        ]
        new_images = []
        new_labels = []
        if self.toaugment:
            for image, label in data:
                for transform in self.transforms:
                    new_images.append(transform(image))
                    new_labels.append(label)
            self.data = list(zip(new_images, new_labels))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image, label = self.data[idx]
        return image, label

In [26]:
# parameters
batch_size = 8
iterations = 12000
workers = 4
print_freq = 100
learning_rate = 0.0001

In [19]:
from torch.utils.data import DataLoader

# Use it when defining your DataLoaders
train_dataloader = DataLoader(
    AugmentedDataset(train_dataset, toaugment=True), 
    batch_size=64, 
    shuffle=True
)

valid_dataloader = DataLoader(
    AugmentedDataset(val_dataset, toaugment=False), 
    batch_size=64, 
    shuffle=False
)

test_dataloader = DataLoader(
    test_dataset, 
    batch_size=64, 
    shuffle=False
)

In [20]:
# CNN implementation for CIFAR-10 classification
import torch.nn as nn
import torch.nn.functional as F

class MyCNN(nn.Module):
    def __init__(self, num_classes=10, dropout_rate=0.5):
        super(MyCNN, self).__init__()
        self.__conv = nn.Sequential(
            # 32x32
            nn.Conv2d(3, 32, kernel_size=5, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout2d(0.1),  # Spatial dropout for conv layers
            nn.MaxPool2d(kernel_size=2, stride=2),  # 16x16
            nn.Conv2d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout2d(0.2),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 8x8
            nn.Conv2d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Dropout2d(0.3),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 4x4
            nn.Conv2d(128, 256, kernel_size=5, padding=2),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Dropout2d(0.4),
            nn.MaxPool2d(kernel_size=2, stride=2)   # 2x2
        )
        self.__flatten = nn.Flatten()
        self.__fc = nn.Sequential(
            nn.Linear(256 * 2 * 2, 150),
            nn.ReLU(),
            nn.Dropout(dropout_rate),  # Dropout in FC layer
            nn.Linear(150, num_classes),
        )

    def forward(self, x):
        conv_result = self.__conv(x)
        flat = self.__flatten(conv_result)
        class_output = self.__fc(flat)
        return class_output

In [27]:
model = MyCNN(num_classes=10, dropout_rate=0.5).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)  # Added L2 regularization

In [22]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for batch in dataloader:
        images, labels = batch
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    epoch_loss = running_loss / len(dataloader)
    return epoch_loss

def train_model(model, train_loader, valid_loader, criterion, optimizer, device, num_epochs, patience=5):
    best_val_loss = float('inf')
    patience_counter = 0
    for epoch in range(num_epochs):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        print(f'Epoch {epoch+1}/{num_epochs}, Training Loss: {train_loss:.4f}')
        
        # validation every epoch
        model.eval()
        valid_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in valid_loader:
                images, labels = batch
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)
                valid_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        valid_loss /= len(valid_loader)
        valid_acc = 100 * correct / total
        print(f'Epoch {epoch+1}/{num_epochs}, Validation Loss: {valid_loss:.4f}, Validation Acc: {valid_acc:.2f}%')
        
        # Early stopping
        if valid_loss < best_val_loss:
            best_val_loss = valid_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

In [28]:
train_model(model, train_dataloader, valid_dataloader, criterion, optimizer, device, num_epochs=100)

Epoch 1/100, Training Loss: 1.6773
Epoch 1/100, Validation Loss: 1.1298, Validation Acc: 59.48%
Epoch 2/100, Training Loss: 1.3611
Epoch 2/100, Validation Loss: 0.9521, Validation Acc: 65.84%
Epoch 3/100, Training Loss: 1.2313
Epoch 3/100, Validation Loss: 0.8348, Validation Acc: 70.33%
Epoch 4/100, Training Loss: 1.1458
Epoch 4/100, Validation Loss: 0.7691, Validation Acc: 72.99%
Epoch 5/100, Training Loss: 1.0830
Epoch 5/100, Validation Loss: 0.7109, Validation Acc: 75.20%
Epoch 6/100, Training Loss: 1.0318
Epoch 6/100, Validation Loss: 0.6755, Validation Acc: 76.26%
Epoch 7/100, Training Loss: 0.9917
Epoch 7/100, Validation Loss: 0.6604, Validation Acc: 77.19%
Epoch 8/100, Training Loss: 0.9585
Epoch 8/100, Validation Loss: 0.6312, Validation Acc: 77.87%
Epoch 9/100, Training Loss: 0.9279
Epoch 9/100, Validation Loss: 0.6169, Validation Acc: 78.43%
Epoch 10/100, Training Loss: 0.9026
Epoch 10/100, Validation Loss: 0.5820, Validation Acc: 79.81%
Epoch 11/100, Training Loss: 0.8788
Ep

KeyboardInterrupt: 

In [29]:
# run test evaluation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch in test_dataloader:
        images, labels = batch
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_acc = 100 * correct / total
print(f'Test Accuracy: {test_acc:.2f}%')

Test Accuracy: 83.69%
